# In this notebook an ANN will be created to estimate the RUL of Batteries
- The work is based on [this paper](https://arxiv.org/pdf/2305.10298)
- The dataset used is [NASA Battery dataset](https://labinfo.ing.he-arc.ch/gitlab/ticc/16TICc19/nasa-battery-dataset/-/tree/614f3518915079a5182655cadc3492973ac90d6c/data/BatteryAgingARC-FY08Q4)

## Step 0: Environment configuration

In [ ]:
#%pip instal numpy; pandas; scipy; matplotlib; seaborn; scikit-learn; keras; tensorflow_cpu; kagglehub

## Step 1: Data collection
### Step 1.1: Data understanding

Generic Dataset Informations¶
Repeated charge and discharge cycles result in accelerated aging of the batteries while impedance measurements provide insight into the internal battery parameters that change as aging progresses.

- Charge profile:
    - The charge profile for all battery tests seems to be identifical.
    - Charging was carried out in a constant current (CC) mode at 1.5A until the battery voltage reached 4.2V and then continued in a constant voltage (CV) mode until the charge current dropped to 20mA.
- Discharge:
    - Discharge profiles were different from battery to battery.
    - Discharge was carried out at a constant current (CC) level of 1-4 A until the battery voltage fell to values such 2.7V, 2.5V, 2.2V and 2.5V.
- Impedance:
    - Impedance measurement was carried out through an electrochemical impedance spectroscopy (EIS) frequency sweep from 0.1Hz to 5kHz.

The experiments were stopped when the batteries reached a given end-of-life (EOL) criteria: for example 30% fade in rated capacity (from 2Ahr to 1.4Ahr). Other stopping criteria were used such as 20% fade in rated capacity. Note that for batteries 49,50,51,52, the experiments were not stop due to battery EOL but because the software has crashed.

Structure of .mat files
- dictionary (loaded mat file)
    - dictionary (e.g. B0005)
        - list (cycle) -> one test per element in the list
            - element of the list (dict) all data for one test of that battery
                - type: operation type, can be charge, discharge or impedance
                - ambient_temperature: ambient temperature (degree C)
                - time: the date and time of the start of the cycle, in MATLAB date vector format
                - data (dict): data structure containing the measurements
                    - data fields with key being measured variable, values the actual records (see below)

| Charge fields | Discharge fields | Impedance fields |
|---|---|---|
| Voltage_measured: Battery terminal voltage (Volts) | Voltage_measured: Battery terminal voltage (Volts) | Frequency: Frequency of impedance measurement (Hz) |
| Current_measured: Battery output current (Amps) | Current_measured: Battery output current (Amps) | Impedance: Battery internal impedance (Ohms) |
| Temperature_measured: Battery temperature (degree C) | Temperature_measured: Battery temperature (degree C) | Real_impedance: Real part of impedance (Ohms) |
| Current_charge: Current measured at charger (Amps) | Current_discharge: Current measured at load (Amps) | Imag_impedance: Imaginary part of impedance (Ohms) |
| Voltage_charge: Voltage measured at charger (Volts) | Voltage_discharge: Voltage measured at load (Volts) | Phase_angle: Phase angle of impedance (degrees) |
| Time: Time vector for the cycle (secs) | Time: Time vector for the cycle (secs) | Time: Time vector for the impedance spectrum (secs) |

### Step 1.2 File conversion from .mat to .csv*
*Firstly the notebook will load data from kaggle where data is already transformed into csv

The csv files will be imported and processed to get data for battery B0005

In [27]:
import pandas as pd
import scipy.io
import kagglehub

# Download latest version
path = kagglehub.dataset_download(handle="patrickfleith/nasa-battery-dataset", output_dir='data')

print("Path to dataset files:", path)


metadata = pd.read_csv('data\\cleaned_dataset\\metadata.csv')
display(metadata.head())

b0005_metadata = metadata[metadata['battery_id'] == 'B0005']
b0005_metadata.info()

b0005_files = b0005_metadata['filename']

b0005_df = pd.DataFrame()

for file in b0005_files:
    path = 'data\\cleaned_dataset\\data\\' + file
    df_temp = pd.read_csv(path)
    b0005_df = pd.concat([b0005_df, df_temp], ignore_index=True)

b0005_df.info()

Path to dataset files: data


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 616 entries, 5120 to 5735
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   type                 616 non-null    str  
 1   start_time           616 non-null    str  
 2   ambient_temperature  616 non-null    int64
 3   battery_id           616 non-null    str  
 4   test_id              616 non-null    int64
 5   uid                  616 non-null    int64
 6   filename             616 non-null    str  
 7   Capacity             168 non-null    str  
 8   Re                   278 non-null    str  
 9   Rct                  278 non-null    str  
dtypes: int64(3), str(7)
memory usage: 48.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 604802 entries, 0 to 604801
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Voltage_measured      591458 non-null  float64
 1   Current_measured  